In [1]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(DelayedArray)
    library(hooke)
    library(splines)
    library(htmlwidgets)
})

In [ ]:
cds_VCS = readRDS("../data/cds_VCS_with_cell_type_labels_E15-15.5_1e-4.RDS")
cds_VCS$Cluster = clusters(cds_VCS)

In [ ]:
## This work is done by visualizing each trajectory and
## reassigning clusters based on the groupings in the UMAP space
manual_filter_df = as.data.frame(colData(cds_VCS)) %>%
    filter(Cluster %in% c(44, 109, 114, 173, 174, 190, 191, 192, 208, 212, 219, 221, 223, 224, 228, 231)) %>%
    count(Cluster, nn_majority_trajectory_umap) %>%
    group_by(Cluster) %>%
    mutate(freq = n / sum(n)) %>% select(-n) %>% arrange(Cluster, -freq) %>%
    mutate(trajectory_refined = nn_majority_trajectory_umap) %>%
    mutate(trajectory_refined = ifelse(Cluster == 44 & trajectory_refined == "Definitive_erythroid", 
                                       "White_blood_cells", trajectory_refined),
          trajectory_refined = ifelse(Cluster == 109, "Intermediate_progenitors", trajectory_refined),
          trajectory_refined = ifelse(Cluster == 114 & trajectory_refined == "CNS_neurons", 
                                      "Intermediate_progenitors", trajectory_refined),
          trajectory_refined = ifelse(Cluster == 173 & trajectory_refined == "Mesoderm", 
                                      "Cardiomyocytes", trajectory_refined),
          trajectory_refined = ifelse(Cluster == 190 & trajectory_refined == "Brain_and_spinal_cord", 
                                      "Eye_and_other", trajectory_refined),
          trajectory_refined = ifelse(Cluster == 192, "Neural_crest_PNS_neurons", trajectory_refined),
          trajectory_refined = ifelse(Cluster %in% c(208, 224, 228), "Epithelium", trajectory_refined),
          trajectory_refined = ifelse(Cluster %in% c(219, 221), "Brain_and_spinal_cord", trajectory_refined),
          trajectory_refined = ifelse(Cluster == 223, "Testis_and_adrenal", trajectory_refined))

In [ ]:
cds_VCS$trajectory_refined = as.data.frame(colData(cds_VCS)) %>%
                    left_join(., refined_df, by = c("Cluster", "nn_majority_trajectory_umap")) %>%
                    pull(trajectory_refined)

In [ ]:
traj_configs <- list(
  "Brain_and_spinal_cord"     = c(res=1e-4,   x=1, y=3),
  "CNS_neurons"               = c(res=8e-5,   x=1, y=3),
  "Definitive_erythroid"      = c(res=1e-4,   x=2, y=3),
  "Endothelium"               = c(res=2e-4,   x=1, y=3),
  "Epithelium"                = c(res=8e-5,   x=1, y=3),
  "Eye_and_other"             = c(res=2e-4,   x=2, y=3),
  "Hepatocytes"               = c(res=1e-4,   x=1, y=3),
  "Intermediate_progenitors"  = c(res=1.5e-4, x=2, y=3),
  "Lung_and_airway"           = c(res=5e-4,   x=1, y=3),
  "Mesoderm"                  = c(res=6e-5,   x=1, y=3),
  "Muscle_cells"              = c(res=7e-5,   x=1, y=3),
  "Neural_crest_PNS_glia"     = c(res=2e-4,   x=1, y=3),
  "Neural_crest_PNS_neurons"  = c(res=1e-4,   x=3, y=2),
  "Olfactory_sensory_neurons" = c(res=1e-4,   x=3, y=2),
  "White_blood_cells"         = c(res=2e-4,   x=3, y=2),
  "Adipocytes"                = c(res=1e-4,   x=3, y=2),
  "B_cells"                   = c(res=1e-4,   x=3, y=2),
  "Ependymal_cells"           = c(res=1e-4,   x=3, y=2),
  "Intestine"                 = c(res=3e-4,   x=3, y=2),
  "Mast_cells"                = c(res=2e-4,   x=3, y=2),
  "Megakaryocytes"            = c(res=1e-4,   x=3, y=2),
  "Oligodendrocytes"          = c(res=1e-4,   x=3, y=2),
  "T_cells"                   = c(res=5e-4,   x=3, y=2),
  "Cardiomyocytes"            = c(res=1e-4,   x=3, y=2),
  "Testis_and_adrenal"        = c(res=1e-3,   x=3, y=2)
)

In [ ]:
for (traj in names(traj_configs)) {
    cat("\nProcessing Trajectory:", traj, "\n")

    res   <- traj_configs[[traj]]["res"]
    x_dim <- traj_configs[[traj]]["x"]
    y_dim <- traj_configs[[traj]]["y"]

    cds_sub <- cds_VCS[, cds_VCS$trajectory_refined == traj]

    cds_sub <- cluster_cells(cds_sub, resolution = res)
    cds_sub$Cluster_sub <- clusters(cds_sub)

    options(repr.plot.width=5, repr.plot.height=5, repr.plot.res=300)
    p <- plot_cells(cds_sub, x = x_dim, y = y_dim,
                  reduction_method = "UMAP", color_cells_by = "cluster", 
                  label_cell_groups = TRUE, cell_size = 0.2, alpha = 0.7) + 
       theme(legend.position = "none") +
       ggtitle(paste0(traj, " (res: ", res, ")"))
    print(p)

    sub_df <- as.data.frame(colData(cds_sub)) %>% select(Cell, Cluster_sub)
    subcluster_df_list[[traj]] <- sub_df
}

In [ ]:
subcluster_df <- bind_rows(subcluster_df_list)

cds_VCS$Cluster_sub = as.data.frame(colData(cds_VCS)) %>%
                        left_join(., subcluster_df, by = "Cell") %>%
                        pull(Cluster_sub)

In [ ]:
saveRDS(cds_VCS, "../data/cds_VCS_with_cell_type_labels_E15-15.5_1e-4_subclustered.RDS")